# Homework 3
## Surface Energy Balance Model

Data are provided from observations taken at the FLUXNET flux tower in the Glacier Lakes Ecosystem Experiments Site (GLEES), located in southern Wyoming, USA.<br>

![GLEES_map.jpg](GLEES_map.jpg)


In this assignment, you will use some of the FLUXNET data as <u>forcing</u> data for models you will run for components of the surface energy budget, and some of it as <u>validation</u> data (and some for both).  We will explore the different capabilities and limitations of both observational data and models. 

The surface heat flux (sensible and latent) measurements use the eddy-covariance approach. Data averaged at 30-minute (half hour, or "HH") intervals spanning a 6-day period in August 2010 are provided in the accompanying file: `FLX_US-GLE_FLUXNET2015_FULLSET_HH.csv`

FLUXNET2015 is a consolodated, standardized and quality-controlled set of data from hundreds of flux towers around the world*. The data are distributed as `.csv` files, which are very easily read in Python using Pandas. Hoewever, that means the only <i>metadata</i> included are the variable names in the column headers. Additional metadata including the units of each variable are available in the
<a href=https://fluxnet.fluxdata.org/data/fluxnet2015-dataset/fullset-data-product/>FLUXNET2015 data documentation</a>. We will use that information to convert the data to NumPy arrays with associated units (using Pint) so that the data will be useable for MetPy.

\* <small>
    Reichstein, M., and Coauthors, 2005: On the separation of net ecosystem exchange into assimilation and ecosystem respiration: Review and improved algorithm. *Global Change Biol.*, **11**, 1424–1439, https://doi.org/10.1111/j.1365-2486.2005.001002.x.<br>
    Vuichard, N., and D. Papale, 2015: Filling the gaps in meteorological continuous data measured at FLUXNET sites with ERA-Interim reanalysis. *Earth Syst. Sci. Data*, **7**, 157–171, https://doi.org/10.5194/essd-7-157-2015.</small>

In [ ]:
# Import useful software packages
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.cbook as cbook

from datetime import datetime
import numpy as np
import pandas as pd
import scipy.stats as stats

import metpy.constants as c
import metpy.calc as calc
from metpy.units import units

import sys, warnings

### Simple Model in Parts -- Redux

We will use again the code from HW-2 for calculating surface evaporation, this time as a means to calculate latent heat flux.

To this, we will additionally use functions to calculate absorbed solar radiation, sensible heat flux, ground heat flux and outgoing longwave radiation.

The FLUXNET data contains all the terms for the surface energy budget, measured by instruments in the field. We will compare our calculations to actual measurements in various ways.

In [ ]:
def sensible_heat_flux(air_temperature,skin_temperature,exchange_coef):
    """
    A simple sensible heat flux model
    
    Required inputs:
        air_temperature  (float or array) = Temperature of air near the surface [K]
        skin_temperature (float or array) = Surface or skin temperature [K]
        exchange_coef    (float or array) = Surface cxchange coefficient for heat [kg/m^2/s]
                
    Outputs:
        shf              (float or array) = Sensible heat flux [W/m^2]
    """
    
    shf  = c.dry_air_spec_heat_press * (skin_temperature.to('K') - air_temperature.to('K')) * exchange_coef
    
    return shf
   
    
#------------------------------------------------------------------------   
def l_vap(temperature):
    """
    Calculates latent heat of vaporization as function of temperature using an accurate empirical equation.
    
    Required inputs:
        temperature   (float or array) = Air temperature [any valid units]
        
    Outputs:
        lambda_v      (float or array) = Latent heat of vaporization [J/kg]
    """
    
    mag_t = temperature.to('celsius').m
    lambda_v = (2500.8-2.36*mag_t+0.0016*mag_t**2-0.00006*mag_t**3)*1000.0 * units('J/kg') # More accurate
    
    return lambda_v


#------------------------------------------------------------------------
def evap_beta(evap_pot,soil_wetness):
    """
    A beta-type evaporation model
    
    Required inputs:
        evap_pot     (float or array) = Potential evaporation rate [units to match output]
        soil_wetness (float or array) = Water content of the soil as a fraction of saturation [-]
        
    Outputs:
        evap         (float or array) = Evaporation rate [units to match input]
    """

    evap = evap_pot * soil_wetness
    
    return evap
    

#------------------------------------------------------------------------
def clau_clap(temperature):
    """
    Returns the rate of change of saturation vapor pressure with temperature at the given 
    temperature, following the Clausius-Clapeyron relationship.
    
    Required inputs:
        temperature (float or array) = Temperature [K] at which to calculate de_s/dT
        
    Outputs:
        desdt       (float or array) = de_s/dT [Pa/K]
    """
  
    # Set some parameters
    R_v = c.water_gas_constant
    #lambda_v = c.water_heat_vaporization
    triple_water = 611.2 * units('Pa')
    
    lambda_v = l_vap(temperature)
    e_s = calc.saturation_vapor_pressure(temperature)
    
    desdt = lambda_v * e_s / (R_v * temperature.to('kelvin')**2)
    
    return desdt


#------------------------------------------------------------------------
def aero_res(density,exchange_coef):
    """
    Returns the aerodynamic resistance given air density and the surface exchange coefficient.
    
    Required inputs:
        density       (float or array) = air density [kg/m^3]
        exchange_coef (float or array) = surface exchange coefficient [kg/m^2/s]
        
    Outputs:
        r_a           (float or array) = aerodynamic resistance [s/m]
    """
    
    r_a = density / exchange_coef
    
    return r_a


#------------------------------------------------------------------------
def priestley_taylor(net_radiation,temperature):
    """
    Calculates potential evaporation based on the Priestley-Taylor equation.
    
    Required inputs:
        net_radiation (float or array) = Net radiation at the surface [W/m^2] (downward positive)
        temperature   (float or array) = Air temperature [K]
        
    Outputs:
        pot_evap      (float or array) = Potential evaporation rate [kg/m^2/s]
    """
    
    # Set some parameters
    mean_slp = 101325 * units('Pa')
    alpha = 1.26  # Priestley-Taylor coefficient [-] (accounts for entrainment of dry air into a growing boundary layer)
    #lambda_v = c.water_heat_vaporization
    lambda_v = l_vap(temperature)
    gamma = (mean_slp*c.dry_air_spec_heat_press)/(lambda_v*c.molecular_weight_ratio) # Psychrometric constant [-]

    # Energy-dependent potential evaporation
    desdt = clau_clap(temperature)
    pot_evap = alpha * desdt * net_radiation / (lambda_v * (desdt + gamma))
    
    return pot_evap


#------------------------------------------------------------------------
def penman(net_radiation,temperature,specific_humidity,density,
           exchange_coef,sfc_sat_spec_humid,pressure=1.0e5*units('Pa')):
    """
    Calculates potential evaporation based on the Penman equation,
        which includes both aerodynamic and energy terms.
    
    Required inputs:
        net_radiation      (float or array) = Net radiation at the surface [W/m^2] (downward positive)
        temperature        (float or array) = Surface air temperature [K]
        specific_humidity  (float or array) = Surface air specific humidity [kg/kg]
        density            (float or array) = Surface air density [kg/m^3]
        exchange_coef      (float or array) = Surface exchange coefficient for heat & moisture [kg/m^2/s]
        sfc_sat_spec_humid (float or array) = Surface saturated specific humidity [kg/kg]
        
    Optional inputs:
        pressure           (float or array) = Surface air pressure [Pa]
        
    Outputs:
        pot_evap           (float or array) = Potential evaporation rate [kg/m^2/s]
    """
    
    # Set some parameters
    mean_slp = 101325 * units('Pa')
    #lambda_v = c.water_heat_vaporization
    lambda_v = l_vap(temperature)
    gamma = (mean_slp*c.dry_air_spec_heat_press)/(lambda_v*c.molecular_weight_ratio) # Psychrometric constant [-]

    desdt = clau_clap(temperature)
    r_a = aero_res(density,exchange_coef)   # Aerodynamic resistance [s/m]
    air_vp = calc.vapor_pressure(pressure,calc.mixing_ratio_from_specific_humidity(specific_humidity))
    sfc_sat_vp = calc.vapor_pressure(pressure,calc.mixing_ratio_from_specific_humidity(sfc_sat_spec_humid))
    
    pot_evap = (desdt*net_radiation + density*c.dry_air_spec_heat_press*(sfc_sat_vp - air_vp)/r_a) / (lambda_v*(desdt + gamma))
    
    return pot_evap


#------------------------------------------------------------------------
def penman_monteith(net_radiation,temperature,specific_humidity,density,
                    exchange_coef,sfc_sat_spec_humid,r_surface,pressure=1.0e5*units('Pa')):
    """
    Calculates evaporation based on the Penman-Monteith equation,
        which includes a surface resistance term to reduce evaporation below the potential rate.
    
    Required inputs:
        net_radiation      (float or array) = Net radiation at the surface [W/m^2] (downward positive)
        temperature        (float or array) = Surface air temperature [K]
        specific_humidity  (float or array) = Surface air specific humidity [kg/kg]
        density            (float or array) = Surface air density [kg/m^3]
        exchange_coef      (float or array) = Surface exchange coefficient for heat & moisture [kg/m^2/s]
        sfc_sat_spec_humid (float or array) = Surface saturated specific humidity [kg/kg]
        r_surface          (float or array) = Surface resistance [s/m]
        
    Optional inputs:
        pressure           (float or array) = Surface air pressure [Pa]
        
    Outputs:
        pot_evap           (float or array) = Potential evaporation rate [kg/m^2/s]
    """
    
    # Set some parameters
    mean_slp = 101325 * units('Pa')
    #lambda_v = c.water_heat_vaporization   
    lambda_v = l_vap(temperature)
    gamma = (mean_slp*c.dry_air_spec_heat_press)/(lambda_v*c.molecular_weight_ratio) # Psychrometric constant [-]
    desdt = clau_clap(temperature)
    r_a = aero_res(density,exchange_coef)   # Aerodynamic resistance [s/m]
    air_vp = calc.vapor_pressure(pressure,calc.mixing_ratio_from_specific_humidity(specific_humidity))
    sfc_sat_vp = calc.vapor_pressure(pressure,calc.mixing_ratio_from_specific_humidity(sfc_sat_spec_humid))
    
    evap = (desdt*net_radiation + density*c.dry_air_spec_heat_press*(sfc_sat_vp - air_vp)/r_a) \
               / (lambda_v*(desdt + (1 + r_surface/r_a)*gamma))
    
    return evap


#------------------------------------------------------------------------
def canopy_resistance(soil_wetness,net_shortwave,temperature,specific_humidity,sfc_temperature,
                      pressure=1.0e5*units('Pa'),min_canopy_res=60*units('s/m')):
    """
    Parameterization of canopy resistance based on four plant stresses:
        • soil water availability
        • light availability
        • vapor pressure deficit (air dryness)
        • temperature 
        Following Sellers and Dorman (1987)
    
    Required inputs:
        soil_wetness       (float or array) = Water content of the soil as a fraction of saturation [-]
        net_shortwave      (float or array) = Net shortwave radiation at the surface [W/m^2] (downward positive)
        temperature        (float or array) = Surface air temperature [K]
        specific_humidity  (float or array) = Surface air specific humidity [kg/kg]
        sfc_temperature    (float or array) = Surface skin temperature [K]

    Optional inputs:
        pressure           (float or array) = Surface air pressure [Pa]
        min_canopy_res     (float or array) = Minimum (optimum unstressed) canopy resistance [s/m]

    Outputs:
        canopy_res         (float or array) = Canopy resistance [s/m]
        
    Notes: 
        Instead of net shortwave radiation, downward shortwave radiation should be used, or more precisely 
        downward PAR (photosynthetically-active radiation), which closely corresponds to visible light.
    """
    # Set some parameters
    min_t = 0 * units('celsius')
    max_t = 45 * units('celsius') # A parabola will be fitted to 0 at these temperatures, and 1 at the midpoint 
    max_vpd = 4000 * units('Pa')
    
    # Soil water stress factor
    f0 = np.maximum(evap_beta(1,soil_wetness), 1.0e-6)

    # Tempeature stress factor
    h1 = 1/((max_t - min_t)/2)**2
    f1 = np.maximum((h1*(sfc_temperature-min_t)*(max_t-sfc_temperature)).m, 0.01)

    # Vapor pressure stress factor
    air_vp = calc.vapor_pressure(pressure,calc.mixing_ratio_from_specific_humidity(specific_humidity))
    sat_vp = calc.saturation_vapor_pressure(temperature)
    vpd = sat_vp - air_vp
    f2 = np.minimum(np.maximum(1 - (vpd/max_vpd).m, 0.05) ,1.0)

    # Light stress factor
    par = 0.5 * net_shortwave  # Approximate PAR
    a3, b3 = -25, 25
    f3 = np.maximum(1 + a3/(b3 + par.m), 0.15) 
    
    canopy_res = min_canopy_res / (f0*f1*f2*f3)
    
    # print(canopy_res,f1,f2,f3)
    
    return canopy_res


#------------------------------------------------------------------------
def stefan_boltzmann(temperature,emissivity=1):
    """
    Calculates emitted radiation (power) as a function of temperature.
    
    Required inputs:
        temperature   (float or array) = Emitting temperature of surface [K]
        
    Optional inputs:
        emissivity    (float or array) = Emitting surface emissivity [0-1] 
        
    Outputs:
        power         (float or array) = Radiated power [W/m^2]
    """
    
    sigma = 5.67e-8 * units('watt/meter**2/K**4')
    
    power = emissivity * sigma * temperature ** 4

    return power


## Read in the data
And do some conversions, because Pandas is great, but it can't handle Pint units. 
So we will put these variable's time series into NumPy arrays with units so we can use MetPy.

In [ ]:
#################################################################################################
#################################################################################################
# Read csv file (csv means "comma-separated values" - basically a table in a text file)
f = pd.read_csv("FLX_US-GLE_FLUXNET2015_FULLSET_HH.csv")
# Note: pd.read_csv reads the file contents into a Pandas DataFrame
#################################################################################################
#################################################################################################

print("Here are all the variables - we will only use a few...")
print(list(f.columns))

# Define a dictionary with units for the FLUXNET variables 
#    Note: First part of column name is the variable name, the text separator is '_'
units_dict = {'TA':'degC','SW':'W/m**2','LW':'W/m**2','VPD':'hPa','PA':'kPa','P':'kg/m**2/s',
              'WS':'m/s','WD':'degree','RH':'Pa/hPa','USTAR':'m/s','NETRAD':'W/m**2',
              'PPFD':'micromole/s/m**2','CO2':'micromole/mole','TS':'degC','SWC':'mm**2/cm**2',
              'G':'W/m**2','LE':'W/m**2','H':'W/m**2','NEE':'micromole/s/m**2',
              'RECO':'micromole/s/m**2','GPP':'micromole/s/m**2'}

n = f.replace(-9999.0,np.nan).to_numpy(dtype='float32')     # Make a 2D numpy array from DataFrame

for j,col in enumerate(f.columns):                   # For each column variable in DataFrame...
    col_vals = n[:,j]                                #     pull out the values for the variable...
    exec(col+" = col_vals")                          #     use the column name for the new variable name...
    for key,val in units_dict.items():               #     see if there are units we can assign...
        if col.split("_")[0] == key:                 #     if the first part of the name is in the units_dict list...
            if col.split("_")[-1] not in ['QC','N','METHOD']: # and not a QC flag or count (end of name) ...
                exec(col+" = "+col+"*units('"+val+"')")  # ...make a new NumPy array as a Pint quantity

# Set timestamps to datetime64            
TIMESTAMP_START = pd.to_datetime(f["TIMESTAMP_START"],format='%Y%m%d%H%M').to_numpy(dtype='datetime64')
TIMESTAMP_END = pd.to_datetime(f["TIMESTAMP_END"],format='%Y%m%d%H%M').to_numpy(dtype='datetime64')
        

### That's a lot of variables!
Most of them have to do with plant ecology [all the ones starting with `NEE` (net ecosystem exchange of carbon), `RECO` (ecosystem respiration; CO2 from land to atmosphere), and `GPP` (gross primary production; CO2 uptake by plants)]. The ones we will use for this assignment are listed below (the suffix `_F_MDS` means _gapfilled using MDS method_):

| Variable Name | What it is | Units |
| --- | --- | :-: |
| `TA_F_MDS` | Air temperature  | $K$ |
| `VPD_F_MDS` | Vapor pressure deficit | $hPa$ |
| `PA` | Atmospheric pressure | $kPa$ |
| `WS` | Wind speed | $m/s$ |
| `USTAR` | Friction velocity | $m/s$ |
| `SW_IN_F_MDS` | Shortwave radiation, incoming | $W/m^2$ |
| `LW_IN_F_MDS` | Longwave radiation, incoming | $W/m^2$ |
| `SW_OUT` | Shortwave radiation, outgoing | $W/m^2$ |
| `LW_OUT` | Longwave radiation, outgoing | $W/m^2$ |
| `NETRAD` | Net radiation | $W/m^2$ |
| `G_F_MDS` | Soil heat flux | $W/m^2$ |
| `LE_F_MDS` | Latent heat flux | $W/m^2$ |
| `H_F_MDS` | Sensible heat flux | $W/m^2$ |
| `LE_CORR` | Latent heat flux, corrected by energy balance closure correction factor | $W/m^2$ |
| `H_CORR` | Sensible heat flux, corrected by energy balance closure correction factor | $W/m^2$ |
| `TS_F_MDS_1` | Soil temperature, 1 = shallowest layer (~5cm deep) | $K$ |
| `SWC_F_MDS_1` | Soil water content, 1 = shallowest layer (~5cm deep) |  $m^3/m^3$ expressed as _%_ |


-----------------------
## A. Apply what you have already learned

In **Part C** of **HW-0** you worked with very similar data from FLUXNET from a different location. Copy your two plotting cells from **C.2** of **HW-0** below, and modify them to plot the 6-days of data from this station, including the energy imbalances using `LE_F_MDS` and `H_F_MDS` versus `LE_CORR` and `H_CORR`. We will use this information later to decide if our model calculations are relatively good (or bad)!

----------------------
## B. Modeling sensible heat flux
First we will see how a calculation of sensible heat flux compares to the values measured in the field. We have a function above to do this:

`sensible_heat_flux(air_temperature,skin_temperature,exchange_coef)`

The variable `TA_F_MDS` is our air temperature. We have two options for the surface skin temperature:
1. Use the upward longwave radiaton (which depends only on radiative skin temperature) 
2. Use the soil temperature in the shallowest layer 

As instruments go, thermometers are more reliable than radiation instruments. However, at 5 _cm_ depth, the temperature variability will be damped quite a bit due to the heat capacity of the soil, and possibly lagged in time due to thermal conductivity. We can use the Stefan-Boltzmann equation $F = \epsilon \, \sigma \, T^4$ to represent skin temperature (as: $\sqrt[4]{F\;/\;\epsilon \, \sigma\;}$), and assuming a reasonable value for emissivity $\epsilon$.

We still need an exchange coefficient. We have what we need to calculate one. 
The data set includes the variable `USTAR`, friction velocity, which is a wind velocity defined by the wind stress over a rough surface.
Aerodynamic resistance can be defined as: $r_a = u\; /\; u_*^2$ where $u$ is just the simple wind velocity. So the exchange coefficient for sensible heatflux $c_p \, \rho\;/\; r_a = c_p \, \rho \, u_*^2 \;/\; u$.
Then we will need to calculate air density $\rho$ from other surface meteorological variables...

In [ ]:
##############################################################
##############################################################
### Parameters for this exercise:
# setting timestep to the observed data interval
timestep = ((TIMESTAMP_START[1]-TIMESTAMP_START[0])/np.timedelta64(1,'s'))*units('s')
##############################################################
##############################################################
epsilon = 0.995                        # Emissivity is never exactly 1.0; tuned to reduce RMSE
c_sb = 5.67e-8 * units('W/m**2/K**4')  # Stefan-Boltzman constant

# Set up empty array to hold the output for later display and comparison
H_model = np.zeros_like(H_F_MDS) * H_F_MDS.u  # Set units the same as H_F_MDS

# Modeling block #########
for i in range(len(TIMESTAMP_START)):  # Run through length of forcing data set
    if i % 48 == 0:               # This if-block prints symbols every hour and day so you can see the progress.
        print("|",end="")
    elif (i % 2 == 0) and (i % 48 != 0):
        print(".",end="")
    
    # We need to calculate the exchange coefficient for heat
    #    ...which means we need density
    #    ...which means we need a better humidity variable than VPD
    #    relting on our handy MetPy "calc" functions:
    svp = calc.saturation_vapor_pressure(TA_F_MDS[i])  # Saturation vapor pressure
    vp = svp - VPD_F_MDS[i]                            # Actual vapor pressure
    mixrat = calc.mixing_ratio(vp,PA_F[i])             # Mixing ratio
    rho = calc.density(PA_F[i],TA_F_MDS[i],mixrat)     # Density
    exch = rho * USTAR[i] * USTAR[i] / WS[i]
    
    # A surface skin temperature from upward longwave radiation
    tskin = np.sqrt(np.sqrt(LW_OUT[i]/(epsilon * c_sb)))

    H_model[i] = sensible_heat_flux(TA_F_MDS[i],tskin,exch)

    
# Plotting block #########
plot_scale = 10
fig = plt.figure(figsize = (2*plot_scale, 1*plot_scale))

plt.plot(TIMESTAMP_START,H_F_MDS,c='black')
plt.plot(TIMESTAMP_START,H_CORR,c='tab:blue')
plt.plot(TIMESTAMP_START,H_model,c='tab:red')
plt.title("Sensible Heat Flux (across 6 days)",fontsize=14)
plt.legend(["Observations","Bowen ratio corrected obs.","Model"],fontsize=12,loc='upper right')
plt.axhline(y=0,ls=":",c="grey")

corr = pd.DataFrame(H_F_MDS.m).corrwith(pd.Series(H_model.m)).loc[0]  # This converting back to Pandas is because it handles missing data greacefully - Numpy doesn't.
corr_c = pd.DataFrame(H_CORR.m).corrwith(pd.Series(H_model.m)).loc[0] 
rmse = np.sqrt(np.square(pd.Series(H_model.m)-pd.Series(H_F_MDS.m)).mean())
rmse_c = np.sqrt(np.square(pd.Series(H_model.m)-pd.Series(H_CORR.m)).mean())
bias = (pd.Series(H_model.m)-pd.Series(H_F_MDS.m)).mean()
bias_c = (pd.Series(H_model.m)-pd.Series(H_CORR.m)).mean()
plt.text(TIMESTAMP_START[0],650,"Model performance:",fontweight='bold',fontsize=14,ha='left')
plt.text(TIMESTAMP_START[0],610,f"Correlation: vs Obs={corr:.2f},  vs Corrected obs={corr_c:.2f}",fontsize=14,ha='left')
plt.text(TIMESTAMP_START[0],570,f"RMS Error:   vs Obs={rmse:.1f},  vs Corrected obs={rmse_c:.2f}",fontsize=14,ha='left')
plt.text(TIMESTAMP_START[0],530,f"Bias:            vs Obs={bias:.1f},  vs Corrected obs={bias_c:.2f}",fontsize=14,ha='left') ;

### Answer a question
This result is not bad - pretty good, actually. There is some missing data on the 6th and possibly a bad data point right at the end of that period. Otherwise, they match up well. But the model seems to simulate a larger range (usually higher maxima in the day time, lower minima at night).
1. Draw on what you have learned about the eddy covariance measurement technique, and how it compares to modeling sensible heat flux as we have done, to explain why we might expect the range of fluxes in observations to be smaller than for the model calculation based on skin and 2m temperature. 

------------------------
## C. Modeling latent heat flux
We have a few choices - let's see how each does.


In [ ]:
##############################################################
##############################################################
### Parameters for this exercise:
# setting timestep to the observed data interval
timestep = ((TIMESTAMP_START[1]-TIMESTAMP_START[0])/np.timedelta64(1,'s'))*units('s')
poros = 0.45 * 100 * units('mm**2/cm**2') # Soil porosity (volumetric soil water content at saturation) as a percentage
##############################################################
##############################################################
epsilon = 0.995                        # Emissivity is never exactly 1.0; tuned to reduce RMSE
c_sb = 5.67e-8 * units('W/m**2/K**4')  # Stefan-Boltzman constant

# Set up empty array to hold the output for later display and comparison
LE_pt_model = np.zeros_like(LE_F_MDS) * LE_F_MDS.u   # Set units the same as LE_F_MDS
LE_pen_model = np.zeros_like(LE_F_MDS) * LE_F_MDS.u  # Set units the same as LE_F_MDS
LE_pm_model = np.zeros_like(LE_F_MDS) * LE_F_MDS.u   # Set units the same as LE_F_MDS

# Modeling block #########
for i in range(len(TIMESTAMP_START)):  # Run through length of forcing data set
    if i % 48 == 0:               # This if-block prints symbols every hour and day so you can see the progress.
        print("|",end="")
    elif (i % 2 == 0) and (i % 48 != 0):
        print(".",end="")
    
    # We need to calculate the exchange coefficient for water vapor (same as for heat?)
    #    ...which means we need density
    #    ...which means we need a better humidity variable than VPD
    #    relting on our handy MetPy "calc" functions:
    svp = calc.saturation_vapor_pressure(TA_F_MDS[i])    # Saturation vapor pressure
    vp = svp - VPD_F_MDS[i]                              # Actual vapor pressure
    mixrat = calc.mixing_ratio(vp,PA_F[i])               # Mixing ratio
    sp_hum = calc.specific_humidity_from_mixing_ratio(mixrat) # Specific humidity
    rho = calc.density(PA_F[i],TA_F_MDS[i],mixrat)       # Density
    exch = rho * USTAR[i] * USTAR[i] / WS[i]             # Exchange coefficient
    
    # A surface skin temperature from upward longwave radiation
    tskin = np.sqrt(np.sqrt(LW_OUT[i]/(epsilon * c_sb)))

    # Need a saturation specific humidity at the surface temperature too.
    mixrat_s = calc.saturation_mixing_ratio(PA_F[i],tskin) # Saturation mixing ratio at skin temperature
    sp_hum_s = calc.specific_humidity_from_mixing_ratio(mixrat_s) # Saturation specific humidity
    
    swi = SWC_F_MDS_1[i]/poros
    netrad = SW_IN_F_MDS[i]+LW_IN_F_MDS[i]-SW_OUT[i]-LW_OUT[i]

    # Calculate latent heat flux for the time step
    pe_pt = priestley_taylor(netrad,TA_F_MDS[i])
    LE_pt_model[i] = evap_beta(pe_pt,swi) * l_vap(TA_F_MDS[i])   # Latent heat flux is evaporation x lambda_v
    
    pe_pen = penman(netrad,TA_F_MDS[i],sp_hum,rho,exch,sp_hum_s,pressure=PA_F[i])
    LE_pen_model[i] = evap_beta(pe_pen,swi) * l_vap(TA_F_MDS[i])

    r_s = canopy_resistance(swi,SW_IN_F_MDS[i]-SW_OUT[i],TA_F_MDS[i],sp_hum,tskin)    
    LE_pm_model[i] = penman_monteith(netrad,TA_F_MDS[i],sp_hum,rho,exch,sp_hum_s,r_s,pressure=PA_F[i]) * l_vap(TA_F_MDS[i])

# Plotting block #########
plot_scale = 10
fig = plt.figure(figsize = (2*plot_scale, 1*plot_scale))

plt.plot(TIMESTAMP_START,LE_F_MDS,c='black')
plt.plot(TIMESTAMP_START,LE_CORR)
plt.plot(TIMESTAMP_START,LE_pt_model)
plt.plot(TIMESTAMP_START,LE_pen_model)
plt.plot(TIMESTAMP_START,LE_pm_model)
plt.ylim(top=1000)
plt.title("Latent Heat Flux (across 6 days)",fontsize=14)
plt.legend(["Observations","Bowen ratio corrected obs.","Priestley-Taylor","Penman","Penman-Monteith"],fontsize=12,loc='upper right')
plt.axhline(y=0,ls=":",c="grey")

corr_pt = pd.DataFrame(LE_F_MDS.m).corrwith(pd.Series(LE_pt_model.m)).loc[0]  # This converting back to Pandas is because it handles missing data greacefully - Numpy doesn't.
corr_pen = pd.DataFrame(LE_F_MDS.m).corrwith(pd.Series(LE_pen_model.m)).loc[0]  
corr_pm = pd.DataFrame(LE_F_MDS.m).corrwith(pd.Series(LE_pm_model.m)).loc[0]  
rmse_pt = np.sqrt(np.square(pd.Series(LE_pt_model.m)-pd.Series(LE_F_MDS.m)).mean())
rmse_pen = np.sqrt(np.square(pd.Series(LE_pen_model.m)-pd.Series(LE_F_MDS.m)).mean())
rmse_pm = np.sqrt(np.square(pd.Series(LE_pm_model.m)-pd.Series(LE_F_MDS.m)).mean())
bias_pt = (pd.Series(LE_pt_model.m)-pd.Series(LE_F_MDS.m)).mean()
bias_pen = (pd.Series(LE_pen_model.m)-pd.Series(LE_F_MDS.m)).mean()
bias_pm = (pd.Series(LE_pm_model.m)-pd.Series(LE_F_MDS.m)).mean()
plt.text(TIMESTAMP_START[0],940,"Model performance:",fontweight='bold',fontsize=14,ha='left')
plt.text(TIMESTAMP_START[0],900,f"Correl: P-T={corr_pt:.2f}, Penman={corr_pen:.2f}, P-M={corr_pm:.2f}",fontsize=14,ha='left')
plt.text(TIMESTAMP_START[0],860,f"RMSE: P-T={rmse_pt:.1f}, Penman={rmse_pen:.1f}, P-M={rmse_pm:.1f}",fontsize=14,ha='left')
plt.text(TIMESTAMP_START[0],820,f"Bias:   P-T={bias_pt:.1f}, Penman={bias_pen:.1f}, P-M={bias_pm:.1f}",fontsize=14,ha='left') ;

### Answer some questions
1. Not much of a contest, but which formulation is performing best and which is worst? Discuss in terms of the 3 skill metrics: temporal correlation, root-mean-square error and bias. 
2. Compare the best latent heat flux model to values of the same skill metrics for sensible heat flux in Part B (which we thought was pretty good) - should we change our opinion about the quality of our sensible heat flux model? Defend your position.
3. There is a _huge_ difference between the results for the Penman and Penman-Monteith formulations, but there is only one extra term in the equation for Penman-Monteith. What is it, and what is it doing to improve the simulation of latent heat flux?
4. The observations suggest there is usually some latent heat flux happening at night most of the time (especially the evening of the 6th and early morning of the 7th), but the Penman-Monteith formulation does not pick it up - it has basically zero latent heat flux at night. Why cannot the Penman-Monteith model pick up these nighttime fluxes? And what happened on the night of the 6th (look at other variables in the observations)?

----------------------
## D. Modeling more than one thing at a time

So far, so good. But rarely in Earth system modeling do we only have to predict or simulate a single quantity. A typical land surface model (LSM) would take as input:
* Near surface meteorology (surface air temperature, humidity, wind speed, pressure, precipitation)
* Incoming radiative energy fluxes (downward shortwave and longwave radiation)

Considering just the water and energy budgets (not carbon), an LSM would then have to calculate:
* The remaining heat fluxes (upward shortwave and longwave radation, sensible and latent heat fluxes, heat flux into the ground)
* Water fluxes (evaporation, transpiration, runoff, baseflow)
* Changes in state of water (freezing, melting)
* Land states (mainly temperatures and soil water content, but also snowpack if it is present)

Calculations would include any intermediate variables - you can see we had to do a fair amount of that above, in what were fairly simple cases.

The problem is that many of these variables are connected - interrelated - but in computer code we have to calculate things one at a time. Let's try upping the ante. 

Given an amount of available energy at each 30-minute time step, if we calculate both _sensible_ and _latent heat fluxes_ with our "best" models and set the _ground heat flux_ to whatever energy is left over to close the budget, what will happen?

In [ ]:
##############################################################
##############################################################
### Parameters for this exercise:
# setting timestep to the observed data interval
timestep = ((TIMESTAMP_START[1]-TIMESTAMP_START[0])/np.timedelta64(1,'s'))*units('s')
poros = 0.45 * 100 * units('mm**2/cm**2') # Soil porosity (volumetric soil water content at saturation) as a percentage
##############################################################
##############################################################
epsilon = 0.995                        # Emissivity is never exactly 1.0; tuned to reduce RMSE
c_sb = 5.67e-8 * units('W/m**2/K**4')  # Stefan-Boltzman constant

# Set up empty array to hold the output for later display and comparison
H_model  = np.zeros_like(LE_F_MDS) * LE_F_MDS.u   # Set units the same as LE_F_MDS
LE_model = np.zeros_like(LE_F_MDS) * LE_F_MDS.u  # Set units the same as LE_F_MDS
G_model  = np.zeros_like(LE_F_MDS) * LE_F_MDS.u   # Set units the same as LE_F_MDS

# Modeling block #########
for i in range(len(TIMESTAMP_START)):  # Run through length of forcing data set
    if i % 48 == 0:               # This if-block prints symbols every hour and day so you can see the progress.
        print("|",end="")
    elif (i % 2 == 0) and (i % 48 != 0):
        print(".",end="")
    
    # We need to calculate the exchange coefficient for water vapor (same as for heat?)
    #    ...which means we need density
    #    ...which means we need a better humidity variable than VPD
    #    relting on our handy MetPy "calc" functions:
    svp = calc.saturation_vapor_pressure(TA_F_MDS[i])    # Saturation vapor pressure
    vp = svp - VPD_F_MDS[i]                              # Actual vapor pressure
    mixrat = calc.mixing_ratio(vp,PA_F[i])               # Mixing ratio
    sp_hum = calc.specific_humidity_from_mixing_ratio(mixrat) # Specific humidity
    rho = calc.density(PA_F[i],TA_F_MDS[i],mixrat)       # Density
    exch = rho * USTAR[i] * USTAR[i] / WS[i]             # Exchange coefficient
    
    # A surface skin temperature from upward longwave radiation
    tskin = np.sqrt(np.sqrt(LW_OUT[i]/(epsilon * c_sb)))

    # Sensible heat flux
    H_model[i] = sensible_heat_flux(TA_F_MDS[i],tskin,exch)

    # Need a saturation specific humidity at the surface temperature too.
    mixrat_s = calc.saturation_mixing_ratio(PA_F[i],tskin) # Saturation mixing ratio at skin temperature
    sp_hum_s = calc.specific_humidity_from_mixing_ratio(mixrat_s) # Saturation specific humidity
    
    swi = SWC_F_MDS_1[i]/poros
    netrad = SW_IN_F_MDS[i]+LW_IN_F_MDS[i]-SW_OUT[i]-LW_OUT[i]

    # Calculate latent heat flux for the time step
    r_s = canopy_resistance(swi,SW_IN_F_MDS[i]-SW_OUT[i],TA_F_MDS[i],sp_hum,tskin)    
    LE_model[i] = penman_monteith(netrad,TA_F_MDS[i],sp_hum,rho,exch,sp_hum_s,r_s,pressure=PA_F[i]) * l_vap(TA_F_MDS[i])
    
    # Ground heat flux as a residual
    G_model[i] = netrad - H_model[i] - LE_model[i]

# Plotting block #########
plot_scale = 10
fig = plt.figure(figsize = (2*plot_scale, 1.5*plot_scale))
spec = fig.add_gridspec(ncols=1, nrows=2, height_ratios=[2,1])
plt.subplots_adjust(hspace=0.10)
ax1 = fig.add_subplot(spec[0,0])
ax2 = fig.add_subplot(spec[1,0])


ax1.plot(TIMESTAMP_START,G_F_MDS,c='black')
ax1.plot(TIMESTAMP_START,G_model,c='tab:orange')
ax1.set_ylim(top=500)
ax1.set_title("Ground Heat Flux (across 6 days)",fontsize=14)
ax1.legend(["Observations","GHF as residual of energy balance"],fontsize=12,loc='upper right')
ax1.axhline(y=0,ls=":",c="grey")

corr = pd.DataFrame(G_F_MDS.m).corrwith(pd.Series(G_model.m)).loc[0]  # This converting back to Pandas is because it handles missing data greacefully - Numpy doesn't.
rmse = np.sqrt(np.square(pd.Series(G_model.m)-pd.Series(G_F_MDS.m)).mean())
bias = (pd.Series(G_model.m)-pd.Series(G_F_MDS.m)).mean()
ax1.text(TIMESTAMP_START[0],455,f"Correlation = {corr:.2f}",fontsize=14,ha='left')
ax1.text(TIMESTAMP_START[0],430,f"RMSE = {rmse:.1f}",fontsize=14,ha='left')
ax1.text(TIMESTAMP_START[0],405,f"Bias = {bias:.1f}",fontsize=14,ha='left')

ax2.step(TIMESTAMP_START,(G_F_MDS.cumsum()*timestep).to('MJ/m**2'),c='black')
ax2.step(TIMESTAMP_START,(np.nan_to_num(G_model).cumsum()*timestep).to('MJ/m**2'),c='tab:orange')
ax2.axhline(y=0,ls=":",c="grey")
ax2.legend(["Observations","Residual of surface energy balance"],fontsize=12,loc='upper left')
ax2.set_title("Cumulative Ground Heat",fontsize=14)

ax2.annotate(f"Final drift:\n{(np.nan_to_num(G_model).sum()*timestep).to('MJ/m**2'):.1f}",
             (TIMESTAMP_START[-1],(np.nan_to_num(G_model).sum()*timestep).to('MJ/m**2').m),
             xytext=(TIMESTAMP_START[-1],(np.nan_to_num(G_model).sum()*timestep).to('MJ/m**2').m-5.0),
             ha='right',fontsize=14,c='tab:orange')

;

### Answer some questions
Yikes! That doesn't look very good. The correlation with observations is pretty low, and there is a strong positive bias that leads to an accumulation of heat in the soil that is about 15x larger than observations!
1. Based on the accumulated energy at the end of the period, and assuming a specific heat per unit volume of $3 \times 10^6$ $J\;K^{-1}\;m^{-3}$ for the soil, how much would all this extra energy warm a layer of soil 10cm thick? _Hint: Use Eq. 2.6 from notes, but pay close attention to the units! The ground heating is not in $J$ but $J\;m^{-2}$. Table C.1 will help you keep the units straight for specific heat by volume._
2. If you have done part 1 correctly, you will get a frightfully large number. In our model we do not have (yet) a prognostic equation for the skin (surface soil) temperature. If we did, the excessive ground heat flux would cause it to rise. But that would start a chain of changes to some of our other fluxes! List those fluxes, what the increasing surface temperature would do to each, and why (based on their formulations).

------------------
## E. Prognostic surface temperature.
Lastly, let's add a prognostic equation for surface temperature and see what happens. Following the previous questions, we will make our soil model a single slab, 10cm thick and having a constant specific heat per unit volume of $3 \times 10^6$ $J\;K^{-1}\;m^{-3}$.

Instead of estimating surface skin temperature from observed LW↑ (often called OLR: outgoing longwave radiation), we will use our predicted value throughout our equations. We will also use it to predict LW↑. Look carefully at the code below to see how it differs from **Part D**.

We will initialize our model with the observed temperature at time 0.

In [ ]:
##############################################################
##############################################################
### Parameters for this exercise:
# setting timestep to the observed data interval
timestep = ((TIMESTAMP_START[1]-TIMESTAMP_START[0])/np.timedelta64(1,'s'))*units('s')
poros = 0.45 * 100 * units('mm**2/cm**2') # Soil porosity (volumetric soil water content at saturation) as a percentage
soil_depth = 0.1 * units['m']             # Depth of surface soil layer
soil_cv = 3.0 * units['MJ/K/m**3']        # Volumetric specific heat of soil layer (constant)
##############################################################
##############################################################

# Set up empty array to hold the output for later display and comparison
H_mod_e  = np.zeros_like(LE_F_MDS) * LE_F_MDS.u   # Set units the same as LE_F_MDS
LE_mod_e = np.zeros_like(LE_F_MDS) * LE_F_MDS.u  
G_mod_e  = np.zeros_like(LE_F_MDS) * LE_F_MDS.u   
LW_mod_e = np.zeros_like(LW_OUT) * LW_OUT.u   
Ts_mod_e = np.zeros_like(TS_F_MDS_1) * TS_F_MDS_1.u   # Set units of temperature
Ts_obs_e = np.zeros_like(TS_F_MDS_1) * TS_F_MDS_1.u   

# A surface skin temperature from upward longwave radiation
epsilon = 0.995                        # Emissivity is never exactly 1.0; tuned to reduce RMSE
c_sb = 5.67e-8 * units('W/m**2/K**4')  # Stefan-Boltzman constant

tskin = (np.sqrt(np.sqrt(LW_OUT[0]/(epsilon * c_sb)))) # Initial surface temperature
 

# Modeling block #########
for i in range(len(TIMESTAMP_START)):  # Run through length of forcing data set
    if i % 48 == 0:               # This if-block prints symbols every hour and day so you can see the progress.
        print("|",end="")
    elif (i % 2 == 0) and (i % 48 != 0):
        print(".",end="")
    
    Ts_obs_e[i] = np.sqrt(np.sqrt(LW_OUT[i]/(epsilon * c_sb))).to('degC')
    
    # We need to calculate the exchange coefficient for water vapor (same as for heat?)
    #    ...which means we need density
    #    ...which means we need a better humidity variable than VPD
    #    relting on our handy MetPy "calc" functions:
    svp = calc.saturation_vapor_pressure(TA_F_MDS[i])    # Saturation vapor pressure
    vp = svp - VPD_F_MDS[i]                              # Actual vapor pressure
    mixrat = calc.mixing_ratio(vp,PA_F[i])               # Mixing ratio
    sp_hum = calc.specific_humidity_from_mixing_ratio(mixrat) # Specific humidity
    rho = calc.density(PA_F[i],TA_F_MDS[i],mixrat)       # Density
    exch = rho * USTAR[i] * USTAR[i] / WS[i]             # Exchange coefficient
    
    # Upward longwave radiation from surface skin temperature
    LW_mod_e[i] = epsilon * c_sb * tskin**4

    # Sensible heat flux based on prognostic surface temperature
    H_mod_e[i] = sensible_heat_flux(TA_F_MDS[i],tskin,exch)

    # Need a saturation specific humidity at the surface temperature too.
    mixrat_s = calc.saturation_mixing_ratio(PA_F[i],tskin) # Saturation mixing ratio at prognostic surface temperature
    sp_hum_s = calc.specific_humidity_from_mixing_ratio(mixrat_s) # Saturation specific humidity
    
    swi = SWC_F_MDS_1[i]/poros
    #netrad = SW_IN_F_MDS[i]+LW_IN_F_MDS[i]-SW_OUT[i]-LW_OUT[i]
    netrad = SW_IN_F_MDS[i]+LW_IN_F_MDS[i]-SW_OUT[i]-LW_mod_e[i] # Using lw_out from prognostic surface temperature

    # Calculate latent heat flux using prognostic surface temperature
    r_s = canopy_resistance(swi,SW_IN_F_MDS[i]-SW_OUT[i],TA_F_MDS[i],sp_hum,tskin)    
    LE_mod_e[i] = penman_monteith(netrad,TA_F_MDS[i],sp_hum,rho,exch,sp_hum_s,r_s,pressure=PA_F[i]) * l_vap(TA_F_MDS[i])
    
    # Ground heat flux as a residual
    G_mod_e[i] = netrad - H_mod_e[i] - LE_mod_e[i]
    # Update the surface temperature for the next time step
    Ts_mod_e[i] = (tskin + G_mod_e[i] * timestep / (soil_cv * soil_depth)).to('degC')
    tskin = Ts_mod_e[i].to('K')

# Plotting block #########
plot_scale = 10
fig = plt.figure(figsize = (2*plot_scale, 2*plot_scale))
spec = fig.add_gridspec(ncols=1, nrows=3, height_ratios=[2,2,1])
plt.subplots_adjust(hspace=0.10)
ax1 = fig.add_subplot(spec[0,0])
ax2 = fig.add_subplot(spec[1,0])
ax3 = fig.add_subplot(spec[2,0])


ax1.plot(TIMESTAMP_START,Ts_obs_e,c='black')
ax1.plot(TIMESTAMP_START,Ts_mod_e,c='tab:red')
ax1.set_title("Surface (skin) temperature (across 6 days)",fontsize=14)
ax1.legend(["Obs $T_s$ (derived from LW↑)","Prognostic $T_s$"],fontsize=12,loc='upper right')

corr = pd.DataFrame(Ts_obs_e.m).corrwith(pd.Series(Ts_mod_e.m)).loc[0]  # This converting back to Pandas is because it handles missing data greacefully - Numpy doesn't.
rmse = np.sqrt(np.square(pd.Series(Ts_mod_e.m)-pd.Series(Ts_obs_e.m)).mean())
bias = (pd.Series(Ts_mod_e.m)-pd.Series(Ts_obs_e.m)).mean()
ax1.text(TIMESTAMP_START[30],19,f"Correlation = {corr:.2f}",fontsize=14,ha='left')
ax1.text(TIMESTAMP_START[30],18.2,f"RMSE = {rmse:.1f}",fontsize=14,ha='left')
ax1.text(TIMESTAMP_START[30],17.4,f"Bias = {bias:.1f}",fontsize=14,ha='left')


ax2.plot(TIMESTAMP_START,G_F_MDS,c='black')
ax2.plot(TIMESTAMP_START,G_mod_e,c='tab:orange')
ax2.set_ylim(top=500,bottom=-500)
ax2.set_title("Ground Heat Flux (across 6 days)",fontsize=14)
ax2.legend(["Observations","Model"],fontsize=12,loc='upper right')
ax2.axhline(y=0,ls=":",c="grey")

corr = pd.DataFrame(G_F_MDS.m).corrwith(pd.Series(G_mod_e.m)).loc[0]  # This converting back to Pandas is because it handles missing data greacefully - Numpy doesn't.
rmse = np.sqrt(np.square(pd.Series(G_mod_e.m)-pd.Series(G_F_MDS.m)).mean())
bias = (pd.Series(G_mod_e.m)-pd.Series(G_F_MDS.m)).mean()
ax2.text(TIMESTAMP_START[30],400,f"Correlation = {corr:.2f}",fontsize=14,ha='left')
ax2.text(TIMESTAMP_START[30],350,f"RMSE = {rmse:.1f}",fontsize=14,ha='left')
ax2.text(TIMESTAMP_START[30],300,f"Bias = {bias:.1f}",fontsize=14,ha='left')

ax3.step(TIMESTAMP_START,(G_F_MDS.cumsum()*timestep).to('MJ/m**2'),c='black')
ax3.step(TIMESTAMP_START,(np.nan_to_num(G_model).cumsum()*timestep).to('MJ/m**2'),c='tab:purple')
ax3.step(TIMESTAMP_START,(np.nan_to_num(G_mod_e).cumsum()*timestep).to('MJ/m**2'),c='tab:orange')
ax3.axhline(y=0,ls=":",c="grey")
ax3.legend(["Observations","From Part D","Model with prognostic $T_s$"],fontsize=12,loc='upper left')
ax3.set_title("Cumulative Ground Heat",fontsize=14)

#ax2.annotate(f"Final drift:\n{(np.nan_to_num(G_model).sum()*timestep).to('MJ/m**2'):.1f}",
#             (TIMESTAMP_START[-1],(np.nan_to_num(G_model).sum()*timestep).to('MJ/m**2').m),
#             xytext=(TIMESTAMP_START[-1],(np.nan_to_num(G_model).sum()*timestep).to('MJ/m**2').m-5.0),
#             ha='right',fontsize=14,c='tab:blue')

;

### Answer some questions
1. The modeled temperature is very close to observations, and certainly not sailing off to the extremely high values you would have predicted from the results in **Part D**. Where has all that heat (energy) gone? _Hint: compare the 6-day mean surface fluxes predicted by this version of the model (sensible, latent and longwave out) to the values from **Part D** (we used different variables names, so they are still there) <u>and observations</u>._ 
2. Although the ground heat flux is still not well correlated with the observations, the bias is now very small (middle panel) and the warm drift has gone away (bottom panel). But we see the surface heat flux fluctuates a lot more than the observations from timestep to timestep (middle panel) and there is a faster accumulation of heat during the morning and loss of heat in the afternoon and evening (bottom panel). What could explain this difference? Give at least <u>two</u> reasons - there are multiple causes! 